In [ ]:
-- 카탈로그 및 스키마 세팅
USE CATALOG acv_adb;
CREATE SCHEMA IF NOT EXISTS gold;

--  테이블 초기화 
DROP TABLE IF EXISTS acv_adb.gold.video_analysis_details;
DROP TABLE IF EXISTS acv_adb.gold.video_analysis_summary;

-- 1 영상 종합 요약 테이블
CREATE TABLE IF NOT EXISTS acv_adb.gold.video_analysis_summary (
    video_id STRING NOT NULL COMMENT '유튜브 영상 고유 ID (PK) ',
    title STRING COMMENT '영상 제목',
    channel_name STRING COMMENT '채널명',
    published_at TIMESTAMP COMMENT '영상 업로드 일시',
    overall_trust_score DOUBLE COMMENT 'LLM 산출 종합 신뢰도 점수',
    trust_level STRING COMMENT '신뢰 등급',
    overall_summary STRING COMMENT '영상 전체 분석 결론',
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP() COMMENT '데이터 적재 시간'
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT '영상 종합 요약 테이블';


-- 2 영상 상세 판별 테이
CREATE TABLE IF NOT EXISTS acv_adb.gold.video_analysis_details (
    claim_id BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL COMMENT '요청 고유 ID (PK)',
    video_id STRING NOT NULL COMMENT '영상 ID (FK)',
    chunk_index INT COMMENT '스크립트 내 문단 순서',
    claim_text STRING COMMENT '검증 대상 주장',
    verification_status STRING COMMENT '검증 결과 상태',
    individual_score DOUBLE COMMENT '개별 점수',
    reason STRING COMMENT '판단 근거',
    source_links ARRAY<STRING> COMMENT '웹 출처'
) USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported',
    'delta.feature.identityColumns' = 'supported'
)
COMMENT '영상 상세 판별 테이블';

-- 제약조건

-- 부모 PK 
ALTER TABLE acv_adb.gold.video_analysis_summary ADD CONSTRAINT summary_pk PRIMARY KEY(video_id);
-- 자식 PK 및 부모 연결(FK)
ALTER TABLE acv_adb.gold.video_analysis_details ADD CONSTRAINT details_pk PRIMARY KEY(claim_id);
ALTER TABLE acv_adb.gold.video_analysis_details ADD CONSTRAINT details_fk FOREIGN KEY(video_id) REFERENCES acv_adb.gold.video_analysis_summary(video_id);


-- 조회 성능 최적화 (Z-ORDER)
-- 특정 영상 ID로 데이터를 빠르게 불러오기 위한 인덱싱 최적화
OPTIMIZE acv_adb.gold.video_analysis_details ZORDER BY (video_id);
OPTIMIZE acv_adb.gold.video_analysis_summary ZORDER BY (video_id);


DESCRIBE EXTENDED acv_adb.gold.video_analysis_summary;

In [ ]:
import json
import re
from pyspark.sql import Row

# 1. 입력된 transcript에서 메타데이터(video_id, title) 추출하기
transcript_text = initial_input["transcript"]

# 정규식으로 title과 source URL 뽑아내기
title_match = re.search(r'title:\s*"([^"]+)"', transcript_text)
video_title = title_match.group(1) if title_match else "Unknown Title"

source_match = re.search(r'source:\s*"https://www.youtube.com/watch\?v=([^"]+)"', transcript_text)
video_id = source_match.group(1) if source_match else "UNKNOWN_ID"

# 2. 결과 데이터 파싱
if "final_analysis" in final_state:
    analysis = final_state["final_analysis"]
    
    # ==========================================
    # [부모 테이블] 요약(Summary) 데이터 준비
    # ==========================================
    summary_data = [Row(
        video_id=video_id,
        title=video_title,
        channel_name="Unknown", # 필요시 추출 로직 추가
        overall_trust_score=float(analysis.get("final_trust_score", 0.0)),
        trust_level=analysis.get("trust_level_indicator", ""),
        overall_summary=analysis.get("conclusion", "")
    )]
    df_summary = spark.createDataFrame(summary_data)
    df_summary.createOrReplaceTempView("temp_summary")

    # ==========================================
    # [자식 테이블] 상세(Details) 데이터 준비
    # ==========================================
    details_data = []
    # 💡 LLM이 뽑은 주장(claim_text)을 가져오기 위해 extracted_claims와 조인
    extracted_claims = {c["claim_id"]: c["claim_text"] for c in final_state.get("extracted_claims", [])}
    
    for item in analysis.get("claim_analysis", []):
        cid = item.get("claim_id")
        
        details_data.append(Row(
            video_id=video_id,
            chunk_index=int(cid.replace("C", "")) if cid.startswith("C") else 0, # C01 -> 1
            claim_text=extracted_claims.get(cid, "주장 텍스트 누락"),
            verification_status=item.get("verification_status", ""),
            individual_score=float(item.get("individual_score", 0.0)),
            # 판단 근거들을 하나의 논리적 서술로 병합
            reason=f"[과학적 근거] {item.get('scientific_evidence', '')}\n[논리 평가] {item.get('logical_argumentation', '')}",
            source_links=[item.get("reference_url", "")] if item.get("reference_url") else []
        ))
        
    if details_data:
        df_details = spark.createDataFrame(details_data)
        df_details.createOrReplaceTempView("temp_details")

    # ==========================================
    # 3. Gold 테이블에 안전하게 적재 (SQL 실행)
    # ==========================================
    
    # 3-1. 부모 테이블 (Summary) - MERGE INTO (있으면 덮어쓰기, 없으면 삽입)
    spark.sql("""
        MERGE INTO acv_adb.gold.video_analysis_summary AS target
        USING temp_summary AS source
        ON target.video_id = source.video_id
        WHEN MATCHED THEN
          UPDATE SET 
            title = source.title,
            overall_trust_score = source.overall_trust_score,
            trust_level = source.trust_level,
            overall_summary = source.overall_summary,
            analyzed_at = CURRENT_TIMESTAMP()
        WHEN NOT MATCHED THEN
          INSERT (video_id, title, channel_name, overall_trust_score, trust_level, overall_summary, analyzed_at)
          VALUES (source.video_id, source.title, source.channel_name, source.overall_trust_score, source.trust_level, source.overall_summary, CURRENT_TIMESTAMP())
    """)
    
    # 3-2. 자식 테이블 (Details) - 기존 데이터 삭제 후 새로 삽입
    # (세부 주장은 내용이 계속 바뀔 수 있고, claim_id가 자동 증가(IDENTITY)이므로 지우고 넣는 것이 깔끔함)
    if details_data:
        spark.sql(f"DELETE FROM acv_adb.gold.video_analysis_details WHERE video_id = '{video_id}'")
        
        # 💡 claim_id는 GENERATED BY DEFAULT AS IDENTITY 이므로 INSERT 목록에서 제외하면 DB가 알아서 번호를 매겨줍니다!
        spark.sql("""
            INSERT INTO acv_adb.gold.video_analysis_details 
            (video_id, chunk_index, claim_text, verification_status, individual_score, reason, source_links)
            SELECT video_id, chunk_index, claim_text, verification_status, individual_score, reason, source_links
            FROM temp_details
        """)
        
    print(f"✅ 분석 결과가 Gold 테이블(Summary & Details)에 완벽하게 적재되었습니다! (Video ID: {video_id})")

else:
    print("❌ final_analysis 결과가 없어 테이블에 적재하지 못했습니다.")